# Week 21, Day 1: The shell

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w021d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import posixpath

here = "/home/you/projects"
print(posixpath.normpath(posixpath.join(here, "data")))
print(posixpath.normpath(posixpath.join(here, "..")))
print(posixpath.normpath(posixpath.join(here, "../photos/./lagos.jpg")))

*Interactive figure on the website: A practice terminal with its own files. Try the tasks below it; each turns green when it's done..*

> **Common mistake.** `rm` deletes **at once and for good**. There's no recycle bin and no "are you sure?". `rm -r` deletes a whole folder tree. Before pressing Enter on any `rm`, read the command again, especially if it has a wildcard like `*`. Many people use `ls` with the same pattern first, to see exactly what would be deleted.

> **Key idea.** The shell works in a current folder. `pwd`, `ls`, `cd` to look and move; `mkdir`, `touch`, `cp`, `mv`, `rm` to change things; `cat` and `head` to read. Paths are absolute (from `/`) or relative (from here), with `~`, `.` and `..`. `rm` is permanent. Use Tab and the up arrow.

### Exercise: Where does it point?

The working directory is `/home/you/projects` and home is `/home/you`. Work out the **absolute** path each of these points to, without running anything.

In [ ]:
a = ___   # data/marks.csv
b = ___   # ..
c = ___   # ../photos/lagos.jpg
d = ___   # ./hello.py
e = ___   # ~/notes.txt

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"/home/you/projects/data/marks.csv\"\n    assert b == \"/home/you\"\n    assert c == \"/home/you/photos/lagos.jpg\"\n    assert d == \"/home/you/projects/hello.py\"\n    assert e == \"/home/you/notes.txt\"")

### Exercise: Resolve a path

Write `resolve(cwd, path, home="/home/you")` that returns the absolute path that `path` points to, given the working directory `cwd`, handling `/` at the start, `~` at the start, `.`, `..` (going up from `/` stays at `/`) and repeated or trailing slashes. Don't use `os.path` or `pathlib`: split on `/` and keep a list of folder names.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert resolve(\"/home/you/projects\", \"data\") == \"/home/you/projects/data\"\n    assert resolve(\"/home/you/projects\", \"../photos/./lagos.jpg\") == \"/home/you/photos/lagos.jpg\"\n    assert resolve(\"/home/you/projects\", \"/etc//hosts/\") == \"/etc/hosts\"\n    assert resolve(\"/home/you/projects\", \"~/notes.txt\") == \"/home/you/notes.txt\"\n    assert resolve(\"/home/you/projects\", \"~\") == \"/home/you\"\n    assert resolve(\"/\", \"../../..\") == \"/\" and resolve(\"/a/b\", \".\") == \"/a/b\" and resolve(\"/a/b\", \"\") == \"/a/b\"\n    assert \"os.path\" not in source and \"pathlib\" not in source and \"posixpath\" not in source")

### Exercise: pathlib

Python's `pathlib` works with paths as objects. Using `PurePosixPath`, set each variable for `p = PurePosixPath("/home/you/projects/data/marks.csv")`.

In [ ]:
from pathlib import PurePosixPath

p = PurePosixPath("/home/you/projects/data/marks.csv")

a = ___   # the file name, "marks.csv"   (p.name)
b = ___   # the name without its ending, "marks"
c = ___   # the ending, ".csv"
d = ___   # the folder it's in, as a string
e = ___   # the path of "report.txt" in that same folder, as a string (use the / operator)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"marks.csv\" and b == \"marks\" and c == \".csv\"\n    assert d == \"/home/you/projects/data\" and e == \"/home/you/projects/data/report.txt\"")

### Exercise: Wildcards

Write `expand(pattern, names)` returning the sorted names that match a shell pattern where `*` matches any run of characters (even none) and `?` matches exactly one character. Write it yourself (no `fnmatch`, `glob` or `re`): a recursive helper `matches(pattern, name)` is the neat way.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    names = [\"notes.txt\", \"todo.txt\", \"hello.py\", \"data.csv\", \"t.py\", \"test.py\"]\n    assert expand(\"*.txt\", names) == [\"notes.txt\", \"todo.txt\"]\n    assert expand(\"t*.py\", names) == [\"t.py\", \"test.py\"]\n    assert expand(\"?.py\", names) == [\"t.py\"]\n    assert expand(\"*\", names) == sorted(names) and expand(\"*.md\", names) == []\n    assert expand(\"*o*\", names) == [\"hello.py\", \"notes.txt\", \"todo.txt\"]\n    assert \"fnmatch\" not in source and \"glob\" not in source and \"import re\" not in source")

### Exercise: Split a command line

The shell splits what you type into words at spaces, except inside quotes. Python's `shlex.split` does the same. Use it to write `parse(line)` returning `(command, options, arguments)`: the first word, a list of the words starting with `-`, and a list of the other words. Return `None` for an empty line.

In [ ]:
import shlex

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert parse(\"ls -l -a projects\") == (\"ls\", [\"-l\", \"-a\"], [\"projects\"])\n    assert parse('cd \"My Documents\"') == (\"cd\", [], [\"My Documents\"])\n    assert parse(\"cp notes.txt 'back up.txt'\") == (\"cp\", [], [\"notes.txt\", \"back up.txt\"])\n    assert parse(\"pwd\") == (\"pwd\", [], []) and parse(\"   \") is None")

### Bug hunt: Going up too far

This path resolver crashes with `IndexError` on `resolve("/a", "../../b")`, because it pops from an empty list. Going up from the top should simply stay at the top, as real shells do. Fix it so it prints `/b`.

In [ ]:
def resolve(cwd, path):
    parts = [p for p in cwd.split("/") if p]
    for seg in path.split("/"):
        if seg == "..":
            parts.pop()
        elif seg and seg != ".":
            parts.append(seg)
    return "/" + "/".join(parts)

print(resolve("/a", "../../b"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"/b\")")

### Bug hunt: Overwritten

This should keep a log of three lines, but the file ends up with only the last one, because each write **replaces** the file, like `>` in the shell. Make it append, like `>>`, so it prints the three lines.

In [ ]:
for event in ["start", "working", "done"]:
    with open("log.txt", "w") as f:
        f.write(event + "\n")

with open("log.txt") as f:
    print(f.read().strip())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"three lines\"\n    reset_files()\n    expect_output(\"start\\nworking\\ndone\")")

### Puzzle: Make a project folder

Arrange the shell commands to make a folder `quiz` in your home folder, move into it, create `main.py` containing one line, and run it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know about the shell.

In [ ]:
a = ___   # the command that shows the working directory
b = ___   # True or False: rm moves files to the recycle bin
c = ___   # after `echo a > f.txt` then `echo b >> f.txt`, how many lines does f.txt have?
d = ___   # after `echo a > f.txt` then `echo b > f.txt`, how many lines does f.txt have?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"pwd\" and b is False and c == 2 and d == 1")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny shell

Write a small shell in Python that works on a pretend file system made of nested dictionaries (a folder is a `dict`, a file is a `str` of its contents). Write `shell(commands)`, which starts with the file system below and the working directory `/home/you`, runs each command line in the list, and returns the list of output lines (one string per line of output; commands with no output add nothing). Support:

- `pwd`; `ls` (names in the working directory, sorted, folders shown with a trailing `/`, all on one line separated by two spaces); `cd path` (with `..`, `~`, absolute and relative paths; `cd` alone goes home);
- `mkdir name`; `echo words... > file` (write the words, joined by spaces, plus a newline, into a file in the working directory); `cat file` (each line of the file as an output line);
- an unknown command gives the line `name: command not found`; a bad `cd` gives `cd: path: No such file or directory`.

```
FS = {"home": {"you": {"notes.txt": "Buy eggs\nCall Ada\n", "projects": {"hello.py": "print('hi')\n"}}}}
```

In [ ]:
import copy

FS = {"home": {"you": {"notes.txt": "Buy eggs\nCall Ada\n", "projects": {"hello.py": "print('hi')\n"}}}}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basics():\n    \"pwd, ls, cd\"\n    assert shell([\"pwd\", \"ls\"]) == [\"/home/you\", \"notes.txt  projects/\"]\n    assert shell([\"cd projects\", \"pwd\", \"ls\", \"cd ..\", \"pwd\", \"cd /\", \"ls\", \"cd\", \"pwd\"]) == [\"/home/you/projects\", \"hello.py\", \"/home/you\", \"home/\", \"/home/you\"]\n    assert shell([\"cd nowhere\", \"pwd\"]) == [\"cd: nowhere: No such file or directory\", \"/home/you\"]\ndef test_files():\n    \"mkdir, echo, cat\"\n    assert shell([\"mkdir tmp\", \"cd tmp\", \"echo hello there > a.txt\", \"ls\", \"cat a.txt\", \"cd ~\", \"cat notes.txt\", \"ls\"]) == [\"a.txt\", \"hello there\", \"Buy eggs\", \"Call Ada\", \"notes.txt  projects/  tmp/\"]\n    assert shell([\"sing\"]) == [\"sing: command not found\"]\n    assert FS[\"home\"][\"you\"].get(\"tmp\") is None, \"run must work on a copy of FS\"")

## Warm-up

Folders nest inside folders, and a **path** says how to get somewhere: `..` means "up one folder" and `.` means "this folder". The shell uses exactly these rules every time you type a command.

## Terminals and shells

A **terminal** is the window; the **shell** is the program inside it that reads what you type, runs it, and shows the result. The shell shows a **prompt** (often ending in `$` or `>`) when it's ready for the next command.

| System | Terminal | Usual shell |
|---|---|---|
| macOS | Terminal (in Applications, Utilities) | zsh |
| Linux | Terminal | bash |
| Windows | Windows Terminal or PowerShell | PowerShell |

A **command** is a program name followed by **arguments**, separated by spaces. Arguments starting with `-` are usually **options** that change what the command does:

```
ls -l projects
```

Here `ls` is the command (list files), `-l` is an option (long form), and `projects` is an argument (which folder to list).

The commands today are the ones used on macOS and Linux. PowerShell on Windows understands most of them too (`ls`, `cd`, `pwd`, `mkdir`, `cp`, `mv`, `rm`, `cat`), with a few differences listed at the end. If you're on Windows, you can also install Git, which comes with "Git Bash", a terminal that behaves like Linux.

## Moving around

The shell is always **in** some folder, called the **working directory**. Every command works relative to it.

- `pwd` (**p**rint **w**orking **d**irectory) shows where you are.
- `ls` lists what's in a folder (this one, unless you name another). `ls -l` gives a longer listing.
- `cd folder` (**c**hange **d**irectory) moves you into a folder. `cd ..` moves up one; `cd` on its own, or `cd ~`, goes home.

Paths come in two kinds:

- **Absolute** paths start at the top of the file system, `/`: `/home/you/projects/hello.py`. They mean the same thing wherever you are.
- **Relative** paths start from where you are: `projects/hello.py`, `../photos`. They're shorter, but depend on the working directory.

Special names: `~` is your home folder, `.` is the current folder, `..` is the folder above.

Try it in the practice terminal below. It has its own pretend files; nothing you type affects your computer. Press Tab to complete a name, and the up arrow to bring back an earlier command.

## Changing things

| Command | Does | Example |
|---|---|---|
| `mkdir name` | make a folder | `mkdir reports` |
| `touch name` | make an empty file (or update its time) | `touch todo.txt` |
| `cat file` | print a file's contents | `cat notes.txt` |
| `head -n 5 file` | the first 5 lines | `head -n 5 data.csv` |
| `cp from to` | copy a file | `cp notes.txt backup.txt` |
| `mv from to` | move or rename | `mv old.txt new.txt` |
| `rm file` | delete a file | `rm backup.txt` |
| `rm -r folder` | delete a folder and everything in it | `rm -r tmp` |
| `echo text` | print text | `echo hello` |

**Wildcards:** `*` matches any run of characters and `?` any single character, so `*.py` means "every file ending in .py". The shell expands the pattern into a list of names **before** the command runs: `rm *.txt` becomes `rm notes.txt todo.txt`.

**Redirection:** `>` sends a command's output into a file instead of the screen, **replacing** the file; `>>` **adds** to the end. `echo "Buy milk" >> todo.txt` appends a line. (A third, `|`, the pipe, sends one command's output into another command; you'll use it tomorrow.)

## Saving typing

- **Tab completion:** type the first few letters of a name and press Tab; the shell completes it (or lists the options). It saves time and avoids typos. Use it constantly.
- **History:** the up arrow brings back earlier commands; `history` lists them.
- **Ctrl+C** stops whatever is running (a program stuck in a loop, say). **Ctrl+L** or `clear` clears the screen.
- **Quotes:** a name with a space in it must be quoted: `cd "My Documents"`. Better still, don't put spaces in names you'll type often.

## On your own computer

Do these now, on your own computer, before the practice exercises:

1. Open your terminal (see the table above). Type `pwd` and press Enter: you're in your home folder.
2. `mkdir atlas`, then `cd atlas`, then `pwd` again.
3. `python3 --version` (on Windows, `python --version` or `py --version`). If it says "not found", install Python from python.org (tick "Add Python to PATH" on Windows) and open a new terminal.
4. `echo "print('Hello from the shell')" > hello.py`, then `cat hello.py`, then `python3 hello.py`.
5. `ls`, then `cd ..`, then `ls` again: you can see your new `atlas` folder from its parent.

**Windows PowerShell differences:** paths use `\` (though `/` usually works too), home is `$HOME` (and `~` works), `rm -r folder` is `rm -Recurse folder`, `touch` is `New-Item name`, `head -n 5 file` is `Get-Content file -TotalCount 5`, and `cat` and `type` both print a file.

## Practice

The practice terminal is good for trying commands; the exercises below build the same ideas in Python, which is also how programs work with paths and files.

## Quiz

## Daily challenge

## Before you go

On your own computer, make a folder for this course's work, `atlas`, and in it a folder for each week from now on (`mkdir week21`). From tomorrow you'll run your Python scripts from the shell there. Tomorrow: running scripts, passing them arguments, and pipes.